# 0.2 — HQ200 complete-car view selection

HQ200 contains several capture circles and some frames crop the car. This notebook uses camera poses to arrange frames by azimuth, divides the orbit into eight rotational sectors, and displays the six best complete-car candidates per sector. The ranking uses the BiRefNet mask boundary margin, foreground coverage, and camera-height consistency.

The pose sequence gives rotational order, but it cannot identify which side is the physical front of every car. Use the per-scene offset/reverse controls to align the displayed order, and override any candidate that still looks incomplete.


In [ ]:
import io
import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
import torch

CPU_ONLY_NOTEBOOK = True
GPU_ATTACHED = torch.cuda.is_available()

print("Runtime check")
print("-" * 50)
print("GPU attached:", GPU_ATTACHED)

if GPU_ATTACHED:
    print("GPU:", torch.cuda.get_device_name(0))
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print(f"GPU memory: {free_bytes / 1024**3:.2f} GB free / {total_bytes / 1024**3:.2f} GB total")

if CPU_ONLY_NOTEBOOK and GPU_ATTACHED:
    raise RuntimeError(
        "This notebook is CPU-only. To conserve Colab GPU availability, "
        "change the Colab hardware accelerator to None, reconnect the CPU "
        "kernel in VS Code, and run the notebook again."
    )

print("Correct CPU runtime: continue with the notebook.")


In [ ]:
from pathlib import Path
from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
if not (DRIVE_MOUNT / "MyDrive").is_dir():
    drive.mount(str(DRIVE_MOUNT))

# Change only this value if the Drive project is moved.
PROJECT_ROOT = DRIVE_MOUNT / "MyDrive" / "ITU" / "3D" / "Thesis"

def find_unique_dir(names, search_roots):
    direct = [root / name for root in search_roots for name in names]
    matches = [p for p in direct if p.is_dir()]
    if not matches:
        for root in search_roots:
            if root.is_dir():
                matches.extend(p for p in root.rglob("*") if p.is_dir() and p.name.casefold() in {n.casefold() for n in names})
    unique = list(dict.fromkeys(p.resolve() for p in matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected one of {names}; found {len(unique)}: {unique}")
    return unique[0]

SEARCH_ROOTS = [PROJECT_ROOT / "data", PROJECT_ROOT]
INDUSTRIAL_ROOT = find_unique_dir(["IndustrialInventory"], SEARCH_ROOTS)
HQ200_ROOT = find_unique_dir(["3DrealCarHQ200", "HQ200"], SEARCH_ROOTS)

# If HQ200 is a wrapper folder, descend to the folder containing capture scenes.
if (HQ200_ROOT / "3DrealCarHQ200").is_dir():
    HQ200_ROOT = HQ200_ROOT / "3DrealCarHQ200"

print("Project:   ", PROJECT_ROOT)
print("Industrial:", INDUSTRIAL_ROOT)
print("3DRealCar: ", HQ200_ROOT)


In [ ]:
PROCESSED_ROOT = PROJECT_ROOT / "data_processed" / "birefnet_original_resolution"
MANIFEST_PATH = PROCESSED_ROOT / "manifest.csv"
SPLIT_ROOT = PROJECT_ROOT / "splits" / "sparse8"
MANUAL_HQ_SELECTION_PATH = SPLIT_ROOT / "hq200_manual_8views.csv"
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)
CANDIDATES_PER_SECTOR = 6
N_SECTORS = 8

manifest = pd.read_csv(MANIFEST_PATH)
hq = manifest[
    manifest.dataset.eq("3DRealCar")
    & manifest.status.isin(["written", "skipped"])
].copy()
# Do not call Path.is_file() for every Drive file. Thousands of individual
# FUSE metadata requests can abort the mounted Drive connection. The
# preprocessing manifest is the source of truth; files are opened lazily.
hq = hq[hq.output_image.notna() & hq.output_mask.notna()].copy()
print("HQ200 scenes:", hq.scene.nunique(), "RGB images:", len(hq))


In [ ]:
def frame_number(path):
    match = re.search(r"frame_(\d+)", Path(path).stem)
    return int(match.group(1)) if match else None

def camera_center(source_path):
    source = Path(source_path)
    metadata_path = source.with_suffix(".json")
    if not metadata_path.is_file():
        return None
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    pose = np.asarray(metadata["cameraPoseARFrame"], dtype=float).reshape(4, 4)
    return pose[:3, 3]

def mask_quality(mask_path, sample_size=(256, 256), threshold=128):
    with Image.open(mask_path) as opened:
        mask = np.asarray(opened.convert("L").resize(sample_size, Image.Resampling.BILINEAR)) >= threshold
    ys, xs = np.where(mask)
    if not len(xs):
        return {"foreground_fraction": 0.0, "margin_fraction": 0.0, "border_contact": 1.0}
    height, width = mask.shape
    margins = np.array([xs.min(), ys.min(), width - 1 - xs.max(), height - 1 - ys.max()], dtype=float)
    border_band = max(2, round(min(width, height) * 0.015))
    border = np.zeros_like(mask)
    border[:border_band] = border[-border_band:] = True
    border[:, :border_band] = border[:, -border_band:] = True
    return {
        "foreground_fraction": float(mask.mean()),
        "margin_fraction": float(margins.min() / min(width, height)),
        "border_contact": float((mask & border).sum() / max(mask.sum(), 1)),
    }

pose_rows = []
for row in hq.itertuples(index=False):
    center = camera_center(row.source)
    if center is None:
        continue
    quality = mask_quality(row.output_mask)
    pose_rows.append({**row._asdict(), "frame_id": frame_number(row.source), "cx": center[0], "cy": center[1], "cz": center[2], **quality})
posed = pd.DataFrame(pose_rows)
print("Images with matching camera poses:", len(posed), "/", len(hq))


In [ ]:
def add_orbit_coordinates(group):
    group = group.copy()
    centers = group[["cx", "cy", "cz"]].to_numpy(float)
    centered = centers - np.median(centers, axis=0)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    plane = vh[:2].T
    coordinates = centered @ plane
    group["orbit_x"] = coordinates[:, 0]
    group["orbit_y"] = coordinates[:, 1]
    group["azimuth"] = np.mod(np.arctan2(coordinates[:, 1], coordinates[:, 0]), 2 * np.pi)
    group["raw_sector"] = np.floor(group.azimuth / (2 * np.pi / N_SECTORS)).astype(int).clip(0, N_SECTORS - 1)
    # Distance away from the median capture plane favours one consistent camera-height circle.
    group["height_deviation"] = np.abs(centered @ vh[2])
    scale = max(float(group.height_deviation.median()), 1e-6)
    group["quality_score"] = (
        6.0 * group.margin_fraction
        - 12.0 * group.border_contact
        - 1.5 * np.abs(group.foreground_fraction - 0.38)
        - 0.15 * group.height_deviation / scale
    )
    return group

posed = pd.concat([add_orbit_coordinates(group) for _, group in posed.groupby("scene", sort=True)], ignore_index=True)

def circular_angle_distance(values, target):
    return np.abs((values - target + np.pi) % (2 * np.pi) - np.pi)

# Targets spread across each sector prevent candidates from clustering at
# almost the same angle. Six candidates provide fine angular choices.
sector_width = 2 * np.pi / N_SECTORS
angular_offsets = np.linspace(-0.30, 0.30, CANDIDATES_PER_SECTOR) * sector_width

candidate_rows = []
for scene, scene_group in posed.groupby("scene", sort=True):
    used_indices = set()
    for sector in range(N_SECTORS):
        sector_center = (sector + 0.5) * sector_width
        selected = []
        for rank, offset in enumerate(angular_offsets, start=1):
            target_angle = np.mod(sector_center + offset, 2 * np.pi)
            pool = scene_group.loc[~scene_group.index.isin(used_indices)].copy()
            pool["target_angle_distance"] = circular_angle_distance(pool.azimuth.to_numpy(), target_angle)

            # Prefer frames close to this angular target. A small overlap
            # beyond the sector boundary gives better diagonal choices.
            nearby = pool[pool.target_angle_distance <= 0.62 * sector_width].copy()
            if nearby.empty:
                nearby = pool.nsmallest(12, "target_angle_distance").copy()

            # Angle is primary; mask quality chooses between nearby frames.
            nearby["candidate_score"] = (
                nearby.quality_score
                - 3.0 * nearby.target_angle_distance / sector_width
            )
            chosen = nearby.sort_values(["candidate_score", "frame_id"], ascending=[False, True]).iloc[0].copy()
            chosen["raw_sector"] = sector
            chosen["candidate_rank"] = rank
            chosen["target_angle"] = target_angle
            chosen["angle_from_target_deg"] = np.degrees(chosen.target_angle_distance)
            chosen["candidate_id"] = f"{scene}__s{sector}__r{rank}__f{int(chosen.frame_id)}"
            selected.append(chosen)
            used_indices.add(chosen.name)
        candidate_rows.append(pd.DataFrame(selected))
candidates = pd.concat(candidate_rows, ignore_index=True)
candidates.to_csv(SPLIT_ROOT / "hq200_view_candidates.csv", index=False)
sector_counts = candidates.groupby(["scene", "raw_sector"]).size().unstack(fill_value=0)
display(sector_counts)
if not (sector_counts > 0).all().all():
    print("WARNING: a scene has an empty pose sector; inspect its trajectory before selection.")


## Candidate sheets

Each figure is one car. Columns follow the camera orbit; rows are candidate ranks 1–6. Prefer an image where the entire car has clear white space around it. The dropdown annotator below is the authoritative selection tool.


In [ ]:
def display_candidate_sheet(scene_candidates):
    scene = scene_candidates.scene.iloc[0]
    fig, axes = plt.subplots(CANDIDATES_PER_SECTOR, N_SECTORS, figsize=(28, 3.2 * CANDIDATES_PER_SECTOR), squeeze=False)
    for sector in range(N_SECTORS):
        sector_rows = scene_candidates[scene_candidates.raw_sector.eq(sector)].sort_values("candidate_rank")
        for rank_index in range(CANDIDATES_PER_SECTOR):
            ax = axes[rank_index, sector]
            if rank_index < len(sector_rows):
                row = sector_rows.iloc[rank_index]
                with Image.open(row.output_image) as opened:
                    ax.imshow(opened.convert("RGB"))
                ax.set_title(
                    f"sector {sector}, candidate {rank_index + 1}\nframe {int(row.frame_id)}\n"
                    f"margin={row.margin_fraction:.3f}, border={row.border_contact:.3f}\n"
                    f"target difference={row.angle_from_target_deg:.1f}°", fontsize=8
                )
                ax.text(0.5, -0.08, row.candidate_id, transform=ax.transAxes, ha="center", va="top", fontsize=6)
            ax.axis("off")
    fig.suptitle(scene, fontsize=15)
    plt.tight_layout()
    plt.show()

SHOW_STATIC_CANDIDATE_SHEETS = False

if SHOW_STATIC_CANDIDATE_SHEETS:
    for scene, scene_candidates in candidates.groupby("scene", sort=True):
        display_candidate_sheet(scene_candidates)
else:
    print("Static 480-image preview skipped. Use the interactive annotator below.")


## Interactive orientation annotator

For each scene, inspect sectors 0–7. Only one sector's six candidates are loaded at a time so the widget remains reliable in VS Code. Label exactly one candidate as each canonical orientation and leave all unused candidates as `IGNORE`. Labels remain in memory while switching sectors; click **Save this scene** before switching cars. The annotations are written to Drive and can be resumed after a disconnect.


In [ ]:
VIEW_LABELS = [
    "FRONT", "FRONT_LEFT", "SIDE_LEFT", "REAR_LEFT",
    "REAR", "REAR_RIGHT", "SIDE_RIGHT", "FRONT_RIGHT",
]
ANNOTATION_PATH = SPLIT_ROOT / "hq200_candidate_annotations.csv"

if ANNOTATION_PATH.is_file():
    saved_annotations = pd.read_csv(ANNOTATION_PATH)
    annotation_state = dict(zip(saved_annotations.candidate_id, saved_annotations.view_label))
    print("Loaded existing annotations:", ANNOTATION_PATH)
else:
    annotation_state = {}

scene_names = sorted(candidates.scene.unique())

def scene_is_complete(scene):
    scene_ids = set(candidates.loc[candidates.scene.eq(scene), "candidate_id"])
    labels = [annotation_state.get(candidate_id, "IGNORE") for candidate_id in scene_ids]
    return all(labels.count(view_label) == 1 for view_label in VIEW_LABELS)

incomplete_scenes = [scene for scene in scene_names if not scene_is_complete(scene)]
initial_scene = incomplete_scenes[0] if incomplete_scenes else scene_names[0]
print(f"Completed scenes: {len(scene_names) - len(incomplete_scenes)}/{len(scene_names)}")
if incomplete_scenes:
    print("Opening first incomplete scene:", initial_scene)
else:
    print("All scenes already have eight saved orientation labels.")

scene_selector = widgets.Dropdown(options=scene_names, value=initial_scene, description="Scene:", layout=widgets.Layout(width="65%"))
sector_selector = widgets.Dropdown(options=list(range(N_SECTORS)), value=0, description="Sector:", layout=widgets.Layout(width="180px"))
candidate_area = widgets.GridBox(layout=widgets.Layout(
    grid_template_columns="repeat(3, minmax(240px, 1fr))",
    grid_gap="12px",
    width="100%",
))
status_output = widgets.Output()
save_button = widgets.Button(description="Save this scene", button_style="success", icon="save")
previous_button = widgets.Button(description="Previous", icon="arrow-left")
next_button = widgets.Button(description="Next", icon="arrow-right")
def thumbnail_bytes(path, size=(320, 220)):
    with Image.open(path) as opened:
        image = opened.convert("RGB")
        image.thumbnail(size, Image.Resampling.LANCZOS)
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=88)
    return buffer.getvalue()

def remember_label(candidate_id, change):
    if change["name"] == "value":
        annotation_state[candidate_id] = change["new"]

def render_page(scene, sector):
    cards = []
    rows = candidates[
        candidates.scene.eq(scene) & candidates.raw_sector.eq(int(sector))
    ].sort_values("candidate_rank")
    for row in rows.itertuples(index=False):
        image_widget = widgets.Image(value=thumbnail_bytes(row.output_image), format="jpeg", layout=widgets.Layout(width="100%"))
        label_widget = widgets.HTML(
            value=f"<b>sector {int(row.raw_sector)} · candidate {int(row.candidate_rank)}</b><br>"
                  f"frame {int(row.frame_id)} · margin {row.margin_fraction:.3f}<br>"
                  f"angle from target {row.angle_from_target_deg:.1f}°<br>"
                  f"<small>{row.candidate_id}</small>"
        )
        dropdown = widgets.Dropdown(
            options=["IGNORE"] + VIEW_LABELS,
            value=annotation_state.get(row.candidate_id, "IGNORE"),
            description="Label:",
            layout=widgets.Layout(width="100%"),
        )
        dropdown.observe(
            lambda change, candidate_id=row.candidate_id: remember_label(candidate_id, change),
            names="value",
        )
        cards.append(widgets.VBox([image_widget, label_widget, dropdown], layout=widgets.Layout(border="1px solid #bbb", padding="6px")))
    candidate_area.children = tuple(cards)
    with status_output:
        clear_output()
        selected = [
            annotation_state.get(candidate_id, "IGNORE")
            for candidate_id in candidates.loc[candidates.scene.eq(scene), "candidate_id"]
        ]
        print(f"{scene} — sector {sector}: showing {len(rows)} candidates.")
        print(f"Orientations currently selected in this scene: {sum(label != 'IGNORE' for label in selected)}/8")
        print("Labels are remembered while changing sectors. Click Save this scene after all 8 sectors are checked.")

def save_current_scene(_=None):
    scene = scene_selector.value
    scene_ids = candidates.loc[candidates.scene.eq(scene), "candidate_id"].tolist()
    labels = {candidate_id: annotation_state.get(candidate_id, "IGNORE") for candidate_id in scene_ids}
    selected = [label for label in labels.values() if label != "IGNORE"]
    missing = [label for label in VIEW_LABELS if selected.count(label) == 0]
    duplicates = [label for label in VIEW_LABELS if selected.count(label) > 1]
    with status_output:
        clear_output()
        if missing or duplicates:
            print("NOT SAVED")
            print("Missing:", missing or "none")
            print("Duplicated:", duplicates or "none")
            return
        annotation_state.update(labels)
        rows = [{"candidate_id": cid, "view_label": label} for cid, label in annotation_state.items()]
        pd.DataFrame(rows).sort_values("candidate_id").to_csv(ANNOTATION_PATH, index=False)
        print("Saved:", scene)
        print("File:", ANNOTATION_PATH)

def move_scene(step):
    index = scene_names.index(scene_selector.value)
    scene_selector.value = scene_names[(index + step) % len(scene_names)]

def scene_changed(change):
    if change["name"] == "value":
        sector_selector.value = 0
        render_page(change["new"], 0)

def sector_changed(change):
    if change["name"] == "value":
        render_page(scene_selector.value, change["new"])

scene_selector.observe(scene_changed, names="value")
sector_selector.observe(sector_changed, names="value")
save_button.on_click(save_current_scene)
previous_button.on_click(lambda _: move_scene(-1))
next_button.on_click(lambda _: move_scene(1))

print(f"Annotation UI: {len(scene_names)} scenes, {len(candidates)} candidates total.")
display(widgets.VBox([
    widgets.HBox([scene_selector, sector_selector, previous_button, next_button, save_button]),
    status_output,
    candidate_area,
]))
# Only six images are embedded at once; 48-image widget payloads can fail
# to render through a hosted Colab connection in VS Code.
render_page(scene_selector.value, sector_selector.value)


## Build the final labeled selection

This cell reads the dropdown annotations. It shows progress without failing when some scenes are unfinished. The final eight-view CSV is written automatically only after every scene has exactly one image for each canonical orientation.


In [ ]:
ANNOTATION_PATH = SPLIT_ROOT / "hq200_candidate_annotations.csv"
OUTPUT_PATH = SPLIT_ROOT / "hq200_manual_8views.csv"
expected_scenes = sorted(candidates.scene.unique())
view_order_map = {label: index for index, label in enumerate(VIEW_LABELS)}

if not ANNOTATION_PATH.is_file():
    selection = candidates.iloc[0:0].copy()
    selection["view_label"] = pd.Series(dtype=str)
    selection["view_order"] = pd.Series(dtype=int)
    print("No saved dropdown annotations yet.")
    print("Use the annotation program above and click 'Save this scene'.")
else:
    annotations = pd.read_csv(ANNOTATION_PATH)
    annotated = annotations[annotations.view_label.isin(VIEW_LABELS)].copy()
    selection = candidates.merge(annotated, on="candidate_id", how="inner")

    label_counts = (
        selection.groupby(["scene", "view_label"])
        .size()
        .unstack(fill_value=0)
        .reindex(index=expected_scenes, columns=VIEW_LABELS, fill_value=0)
    )
    display(label_counts.assign(valid=label_counts.eq(1).all(axis=1)))

    duplicate_scenes = label_counts.index[label_counts.gt(1).any(axis=1)].tolist()
    if duplicate_scenes:
        raise ValueError(f"Duplicate orientation labels found in scenes: {duplicate_scenes}")

    completed_scenes = label_counts.index[label_counts.eq(1).all(axis=1)].tolist()
    incomplete_scenes = sorted(set(expected_scenes) - set(completed_scenes))
    selection = selection[selection.scene.isin(completed_scenes)].copy()
    selection["view_order"] = selection.view_label.map(view_order_map)
    selection = selection.sort_values(["scene", "view_order"])

    if not selection.empty:
        display(selection[["scene", "view_label", "view_order", "candidate_id", "frame_id", "margin_fraction", "border_contact", "output_image"]])

    if incomplete_scenes:
        print(f"Completed {len(completed_scenes)}/{len(expected_scenes)} scenes.")
        print("Still annotate and save:")
        for scene in incomplete_scenes:
            print(" -", scene)
        print("The final selection file was not written yet.")
    else:
        final_counts = selection.groupby("scene").size()
        assert final_counts.eq(8).all(), "Every scene must have exactly eight labeled views."
        selection.to_csv(OUTPUT_PATH, index=False)
        print("All scenes complete. Saved final selection:", OUTPUT_PATH)


In [ ]:
def display_final_selection(selection):
    if selection.empty:
        print("No completed scenes to display yet. Save annotations in the dropdown program first.")
        return
    scenes = sorted(selection.scene.unique())
    fig, axes = plt.subplots(len(scenes), 8, figsize=(28, 3.2 * len(scenes)), squeeze=False)
    for row_index, scene in enumerate(scenes):
        rows = selection[selection.scene.eq(scene)].sort_values("view_order")
        for column_index, (_, row) in enumerate(rows.iterrows()):
            with Image.open(row.output_image) as opened:
                axes[row_index, column_index].imshow(opened.convert("RGB"))
            orientation = row.get("view_label", f"view {column_index + 1}")
            axes[row_index, column_index].set_title(f"{orientation} | frame {int(row.frame_id)}", fontsize=8)
            axes[row_index, column_index].axis("off")
        axes[row_index, 0].set_ylabel(scene, rotation=0, ha="right", va="center", labelpad=85, fontsize=8)
    plt.suptitle("Final HQ200 selection — 8 rotationally ordered complete-car views per scene")
    plt.tight_layout()
    plt.show()

display_final_selection(selection)
